# Finsheild Phase 1 — Dataset pipeline (Colab-ready)
Reproducible loader / preprocessing / splits on the Kaggle Credit Card Fraud dataset. No LLM weights. CPU-only, but runs on GPU runtime as well.

In [ ]:
# Cell 1 — Hardware detection (python, torch, CUDA, GPU mem)
import sys, platform, torch
print(f"Python {sys.version}")
print(f"Platform {platform.platform()}")
print(f"Torch {torch.__version__ if 'torch' in sys.modules else 'not installed yet'}")
try:
    import torch
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        props = torch.cuda.get_device_properties(0)
        print(f"GPU mem: {props.total_memory/1e9:.1f} GB")
    else:
        print("Running CPU-only (expected for Phase 1)")
except Exception as e:
    print(f"Torch check skipped: {e}")
    print("Running CPU-only")


In [ ]:
# Cell 2 — Install deps (Colab-friendly)
!pip install -r requirements-colab.txt  2>&1 | tail -n 20
# Fallback if file not found when running from different cwd:
# !pip install pandas scikit-learn pyyaml matplotlib seaborn joblib kagglehub opendatasets tqdm


In [ ]:
# Cell 3 — Download dataset (Kaggle primary, synthetic fallback)
# Option A: KaggleHub (requires KAGGLE_USERNAME/KAGGLE_KEY or kaggle.json — set as Colab Secrets)
import os, pathlib
from pathlib import Path
print("Attempting KaggleHub download...")
try:
    import kagglehub
    dest = Path("data/raw/creditcard.csv")
    if not dest.exists():
        cache = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
        import shutil
        for p in Path(cache).rglob("creditcard.csv"):
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(p, dest)
            print(f"Downloaded to {dest} via kagglehub")
            break
    else:
        print(f"Already exists at {dest}")
except Exception as e:
    print(f"KaggleHub failed: {e}")
    print("Fallback: run synthetic — !python scripts/download_dataset.py --synthetic")
    # Synthetic fallback for Colab without creds:
    # !python scripts/download_dataset.py --synthetic
    print("Or set Colab Secrets: KAGGLE_USERNAME / KAGGLE_KEY and restart runtime")


In [ ]:
# Cell 4 — Alternative download via opendatasets (also needs Kaggle creds)
# !pip install opendatasets -q
# import opendatasets as od
# od.download('https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud', data_dir='data/raw/_tmp')
# Direct URL fallback (if dataset is mirrored): document in docs/dataset.md


In [ ]:
# Cell 5 — Loader / preprocessing / splits demo (must NOT leak)
from finsheild.data.loader import load_raw
from finsheild.data.splits import make_splits, save_splits
from finsheild.data.preprocessing import FraudPreprocessor, preprocess_splits
import pandas as pd

df = load_raw("data/raw/creditcard.csv")
print(df.head())
print(df["Class"].value_counts())

train, val, test = make_splits(df, test_size=0.15, val_size=0.15, random_state=42)
print(f"\nSplits: train {train.shape}, val {val.shape}, test {test.shape}")

# Leakage-safe preprocessing: fit ONLY on train
pre = FraudPreprocessor(scale_features=["Amount", "Time"])
train_t = pre.fit_transform_train(train)
val_t = pre.transform(val)
test_t = pre.transform(test)
print("\nScaler means:", pre.scaler.mean_)
print("Train Amount mean after scaling:", train_t["Amount"].mean())

# Save processed + scaler (gitignored)
save_splits(train_t, val_t, test_t, out_dir="data/processed", fmt="csv")
pre.save("data/processed/scaler.joblib")
print("Saved processed splits + scaler")


In [ ]:
# Cell 6 — EDA quick checks (shape, dtypes, missing, class dist, Amount/Time by class)
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
print(f"Shape: {df.shape}")
print(df.dtypes)
print(f"Missing: {df.isnull().sum().sum() == 0}")
print(f"Duplicated rows: {df.duplicated().sum()}")
print(df["Class"].value_counts(normalize=True))

Path("evaluation/figures").mkdir(parents=True, exist_ok=True)
plt.figure(figsize=(4,3))
sns.countplot(x="Class", data=df)
plt.title("Class distribution")
plt.savefig("evaluation/figures/class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1,2, figsize=(10,4))
for ax, col in zip(axes, ["Amount","Time"]):
    df.boxplot(column=col, by="Class", ax=ax)
    ax.set_title(col)
plt.suptitle("")
plt.tight_layout()
plt.savefig("evaluation/figures/amount_time_by_class.png", dpi=150, bbox_inches="tight")
plt.show()
# Also write metrics for Drive sync (if not already by previous cells)
import json
from pathlib import Path as _P
_P("evaluation").mkdir(parents=True, exist_ok=True)
# metrics.json already created by previous demo; if missing, create minimal placeholder
if not _P("evaluation/metrics.json").exists():
    json.dump({"dataset_shape": list(df.shape), "note": "Phase 1 EDA"}, open("evaluation/metrics.json","w"), indent=2)
print("Local artifacts ready: evaluation/metrics.json, evaluation/reports/dataset_report.md, figures/")


## Cell 7 — Auto-save to Drive (optional, won't fill your laptop disk)
This cell copies **only tiny artifacts** (`metrics.json` ~2 KB, `dataset_report.md` ~5 KB, `*.png` ~200 KB, `scaler.joblib` ~5 KB) to your **Google Drive** — which is cloud storage, *not* your laptop's SSD. `data/raw/creditcard.csv` (~150 MB) and `data/processed/*.csv` (~30 MB) are **not** auto-copied unless you flip `COPY_PROCESSED=True`. Toggle `CLEAN_LOCAL` to delete local copies after upload if you're tight on disk. If you're not in Colab (e.g. local VS Code without Drive), the cell just skips.


In [ ]:
# Cell 7 — Auto-save tiny artifacts to Drive (cloud, not local disk) + optional cleanup
from pathlib import Path
import shutil, json, os, sys
DRIVE_ROOT = Path("/content/drive/MyDrive/Finsheild")  # change if you want another folder
COPY_PROCESSED = False  # set True to also copy data/processed/*.csv (heavier, ~30 MB)
CLEAN_LOCAL = False     # set True to delete local data/processed/*.csv after copy to free disk
SAVE_RAW = False        # keep False — raw is 150 MB and gitignored; Drive copy is optional
print(f"DRIVE_ROOT={DRIVE_ROOT} | COPY_PROCESSED={COPY_PROCESSED} | CLEAN_LOCAL={CLEAN_LOCAL} | SAVE_RAW={SAVE_RAW}")
try:
    from google.colab import drive  # only exists inside Colab runtime
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not in Colab runtime — skipping Drive mount (nothing to fill your laptop)")
if IN_COLAB:
    if not Path("/content/drive/MyDrive").exists():
        print("Mounting Drive...")
        drive.mount("/content/drive")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    # show local disk before copy
    try:
        import subprocess; print(subprocess.check_output(["df","-h","/content"], text=True).splitlines()[-1])
    except Exception: pass
    # files to copy: tiny artifacts only
    to_copy = ["evaluation/metrics.json", "evaluation/reports/dataset_report.md", "evaluation/figures/class_distribution.png", "evaluation/figures/amount_time_by_class.png", "data/processed/scaler.joblib"]
    if COPY_PROCESSED:
        to_copy += ["data/processed/train.csv","data/processed/val.csv","data/processed/test.csv"]
    if SAVE_RAW:
        to_copy.append("data/raw/creditcard.csv")
    copied = []
    for src in to_copy:
        s = Path(src)
        if not s.exists():
            print(f"skip missing {src}")
            continue
        d = DRIVE_ROOT / src
        d.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(s, d)
        copied.append(str(d))
        print(f"copied {s} ({s.stat().st_size/1024:.1f} KB) → {d}")
    # lightweight marker so next phases know where Drive artifacts live
    (DRIVE_ROOT / "_last_sync.json").write_text(json.dumps({"copied": copied}, indent=2))
    print(f"Drive sync done: {len(copied)} files → {DRIVE_ROOT}")
    if CLEAN_LOCAL and COPY_PROCESSED:
        for p in Path("data/processed").glob("*.csv"):
            p.unlink(); print(f"cleaned local {p} to free disk")
    try:
        import subprocess; print(subprocess.check_output(["df","-h","/content"], text=True).splitlines()[-1])
    except Exception: pass
    print("Local disk not filled: only tiny artifacts copied to cloud Drive. Raw stays gitignored.")
else:
    print("Local run — artifacts already under evaluation/ and data/processed/ (gitignored raw/processed). No Drive needed.")
    print("To free disk later: rm -rf data/raw data/processed evaluation/figures (they regenerate)")
